# Integrated Intelligent Systems: Tutorial 3 - Body Motion Problem (Scientific Object) (3 Pts)

This notebook introduces a formulation body motion problem, central to designing and implementing Cognition-Enabled Robot Agents, as a description logic system.

## Part 1: Robot-World Manipulation System as Scientific Object (E1–E6) (1.5 Pt)

**Title:** Formalizing the Robot Agent: From Architecture to Description Logic Reasoners

![Alternative Text](RWMS.png)

### Scenario Background
An autonomous robotic agent (the PR2 in the EASE apartment) is deployed to execute a manipulation task ("Serve Eva a drink"). To validate this agent scientifically, we must formalize the system as a **Robot-World Manipulation System (RWMS)**, separate the controlling subsystem from the controlled body-world system, define trajectories as trees, and assess runs against an external problem standard using a description logic reasoner.

---

### Multi-Question Student Task Sheet (0.25 Pt x 6)

1. **E1 · Architectural Cut & Subsystem Separation**
   - In the graphical representation of the RWMS, a strict architectural "cut" is placed between the manipulation executive and the body-world system ($[B \bowtie W]$). Explain the nature of this cut: why is the manipulation executive classified as a `dul:Agent` (non-physical program) while the body-world system is classified under physical objects, and how do they interact exclusively across the boundary?

2. **E2 · Forward-Branching Time & World Choices**
   - During an operation on a fridge door or container seal, the same action token (e.g., a position-controlled pull) can lead to completely divergent physical outcomes depending on world choices (e.g., seal releases vs. seal holds tight). Draw or describe the structure of the resulting courses, and explain why the possible courses form a tree that branches *only forward* from the initial state $x_0$.

3. **E3 · Trajectory Classification & Priority Hierarchies**
   - The problem standard $P$ sorts every physical trajectory into one of four classes: **SOL** (Solution), **REPD** (Repaired), **NON** (Nonachievement), and **PROH** (Prohibited). Consider two runs that both successfully place the drink within Eva's reach, but where Run A tears off the container handle (`P4` violation) while Run B maintains licensed contact. Classify both runs and explain why the classifier grades the *entire course* rather than just the final end-state.

4. **E4 · Description Logic Partonomy & DUL Constraints**
   - When formalizing the RWMS in description logic under DOLCE Ultra Lite (DUL), we define an RWMS as having *exactly one* executive and *exactly one* body-world system (`RWMS = (=1 hasComponent.ManipExecutive) ⊓ (=1 hasComponent.BodyWorldSystem)`). Using the transitivity of `dul:isPartOf`, explain how a reasoner (such as HermiT) automatically entails that the PR2's head camera is a component of the RWMS without explicitly hardcoding every part-whole link.

5. **E5 · Ontological Inconsistency & Entailment Limits**
   - An inconsistency in description logic serves as a strict analytical tool. Formulate a competency question that results in an ontological inconsistency when attempting to classify the *manipulation problem itself* as a physical component inside the RWMS. Furthermore, explain why numerical physical laws (such as Coulomb friction $2 \cdot \mu \cdot F \ge m \cdot (g + a)$) must be handled outside description logic by the semantic digital twin's causal model.

6. **E6 · Strength vs. Truth in Agent Validation**
   - The design thesis dictates that an agent decides within its believed causality, which can be tested for logical *strength* via competency queries and for *truth* via physical experiments. Compare how an axiom set that lacks a friction law ($\text{K}_0$) performs on competency queries versus an axiomatization containing explicit dynamical laws ($\text{K}_1/\text{K}_2$), and state what kind of test is required to validate the actual physical truth of those beliefs.

### Answers (Part 1)

**E1. Architectural Cut & Subsystem Separation**

The architectural cut divides the RWMS into the **controlling subsystem** (the manipulation executive, which *reads, decides, and commits* after receiving a task request like "serve Eva a drink") and the **controlled subsystem** (the coupled body-world system $[B \bowtie W]$, consisting of the PR2 robot body $B$ and the kitchen world $W$, which are coupled through sensing/perceiving, physical contact and force/acting, and the world's own dynamics/state changes).

In DOLCE Ultra Lite (DUL), the manipulation executive is modeled under `dul:Agent` (a non-physical computational / information program) because it only processes information, internal beliefs, and control decisions—it has no physical mass, spatial geometry, or direct mechanical contact with the kitchen. In contrast, $[B \bowtie W]$ is classified under `dul:PhysicalObject` entities (the PR2 body, arms, grippers, sensors, the fridge door, drink container, and kitchen furniture) that occupy 3D space and evolve under continuous physical laws.

Because of this strict cut, the two subsystems interact **exclusively** across the boundary interface via two signals:
- **Action tokens** (Controlling $\to$ Controlled): motor/control commands sent from the manipulation executive to the robot body $B$.
- **Observations** (Controlled $\to$ Controlling): sensor readings/percepts returned from the body-world system $[B \bowtie W]$ back to the manipulation executive.

---

**E2. Forward-Branching Time & World Choices**

![Forward-branching state tree of courses](E2_course_tree.png)

**Structure of the Resulting Courses:**
A course $c = (x_0, x_1, x_2, \dots)$ is a sequence of physical states in $[B \bowtie W]$ starting from a single initial state $x_0$ (the root of the tree, where the PR2 gripper grasps the fridge handle or container seal). When the manipulation executive commits to an action token (e.g., a position-controlled pull driving the system to state $x_1$), the subsequent transition is not uniquely determined by the command alone. Instead, it branches into multiple distinct physical continuations depending on the environment's response (**world choices**):
- **Course 1 ($x_0 \to x_1 \to x_{2a} \to x_{3a}$):** *World choice 1* — the magnetic/vacuum seal releases smoothly under the applied pull force, the door/container opens as intended, and the drink is served (**SOL**).
- **Course 2 ($x_0 \to x_1 \to x_{2b} \to x_{3b} \text{ or } x_{3c}$):** *World choice 2* — the seal holds tight and exceeds static friction at the fingertips, causing the gripper to slip off the handle without breaking it. From $x_{2b}$, the executive can make a subsequent **agent choice** to either re-grasp and retry ($x_{3b}$, **REPD**) or abort safely ($x_{3c}$, **NON**).
- **Course 3 ($x_0 \to x_1 \to x_{2c} \to x_{3d}$):** *World choice 3* — the seal holds tight while the position controller builds up excessive pulling force that exceeds the structural limit of the container, tearing off the handle (`P4` violation, **PROH**).

**Why Courses Form a Tree That Branches Only Forward from $x_0$:**
1. **Fixed Past (Single Path to the Root):** Up to any present state $x_t$, the sequence of states that already occurred ($x_0 \to x_1 \to \dots \to x_t$) is historically realized and cannot change. Because physical time and causality are irreversible, distinct trajectories never merge backward in time, giving every state a unique ancestor path back to $x_0$.
2. **Open Future (Agent & World Non-Determinism):** Looking forward from any state $x_t$, multiple futures remain physically possible due to unmodeled physical parameters in $[B \bowtie W]$ (variations in seal adhesion, surface friction $\mu$, contact alignment, or material fatigue) combined with future agent decisions. Each resolution of these uncertainties splits the trajectory into new child branches moving strictly forward in time.

---

**E3. Trajectory Classification & Priority Hierarchies**

The problem standard $P$ partitions all physical courses into four mutually exclusive classes ordered by a strict priority hierarchy (**PROH > NON > REPD > SOL**):
- **PROH (Prohibited):** Violates a safety or physical integrity constraint (e.g., damaging an object or human) at any point during the run.
- **NON (Nonachievement):** Terminates safely without violating any prohibition rules, but fails to achieve the target goal state.
- **REPD (Repaired):** Encounters an intermediate execution failure (such as a harmless slip) that the executive detects and recovers from to ultimately reach the goal safely.
- **SOL (Solution):** Achieves the goal state cleanly without any prohibition violations or need for repair.

Based on this hierarchy:
- **Run A:** Classified as **PROH** (Prohibited). Even though the drink ends up within Eva's reach, tearing off the container handle violates the physical safety/integrity constraint (`P4`). Because `PROH` sits at the top of the priority hierarchy, breaking a safety constraint at any point immediately classifies the entire run as prohibited regardless of the successful end-state.
- **Run B:** Classified as **SOL** (Solution) (or **REPD** if it had a minor slip and recovered), because it reaches the goal state while keeping licensed, non-destructive contact throughout the entire execution.

The classifier has to grade the **entire course** rather than just checking the final end-state because safety, licensed contact, and avoiding damage are path invariants (temporal constraints on *how* the task is executed at every intermediate state and transition). If we only checked the final snapshot ("is the drink next to Eva?"), a destructive trajectory that broke the fridge door or tore off handles along the way would be falsely graded as a valid solution (`SOL`). Furthermore, distinguishing between a clean run (**SOL**) and a recovered run (**REPD**) is impossible without inspecting the full intermediate trajectory history.

---

**E4. Description Logic Partonomy & DUL Constraints**

In DUL (and OWL 2 DL), `hasComponent` is defined as a direct (non-transitive) sub-property of `dul:hasPart` (and its inverse `isComponentOf` is a sub-property of `dul:isPartOf`, i.e., `isComponentOf ⊑ dul:isPartOf`). This allows OWL 2 DL to use qualified cardinality restrictions like `RWMS = (=1 hasComponent.ManipExecutive) ⊓ (=1 hasComponent.BodyWorldSystem)` on `hasComponent` while declaring the super-property `dul:isPartOf` (and `dul:hasPart`) as **transitive** (`Trans(dul:isPartOf)`).

In our ontology we only need to assert the local part-whole links at each structural level:
1. `HeadCamera` `dul:isPartOf` `PR2Body` (or via `isComponentOf ⊑ dul:isPartOf`)
2. `PR2Body` `dul:isPartOf` `BodyWorldSystem`
3. From the RWMS definition, the `BodyWorldSystem` is a component of the `RWMS` (`isComponentOf`), which entails `BodyWorldSystem` `dul:isPartOf` `RWMS`.

Because `dul:isPartOf` is transitive ($A \text{ isPartOf } B \land B \text{ isPartOf } C \Rightarrow A \text{ isPartOf } C$), a DL reasoner like HermiT chains these three facts together automatically: from (1) and (2) it infers that `HeadCamera` is part of `BodyWorldSystem`, and combining that with (3) it entails that `HeadCamera` is part of the `RWMS` without us having to manually hardcode direct links from every individual sensor or joint to the RWMS.

---

**E5. Ontological Inconsistency & Entailment Limits**

- **Competency Question for Inconsistency:** *"Is the manipulation problem $P$ ('Serve Eva a drink') a physical component of the Body-World System inside the RWMS?"*
  - In the ontology, the manipulation problem $P$ is a normative task specification, so it is classified under `dul:Description` (a subclass of `dul:SocialObject` / non-physical information entity). Meanwhile, all components of the `BodyWorldSystem` must be physical objects (`BodyWorldSystem ⊑ ∀hasComponent.PhysicalObject`). Since `dul:SocialObject` (including `dul:Description`) and `dul:PhysicalObject` are strictly **disjoint** in DUL (`Description ⊓ PhysicalObject ⊑ ⊥`), asserting that $P$ is a component of the `BodyWorldSystem` forces $P$ to belong to two disjoint classes simultaneously ($P : \bot$), which causes HermiT to report an **ontological inconsistency**. Conceptually, this confirms that the problem standard $P$ is an external evaluative standard used to judge the RWMS, not a physical part inside it.
- **Why Numerical Physical Laws Live Outside DL:**
  - Description Logics are decidable fragments of first-order logic designed for taxonomic classification and relational reasoning over discrete concepts and roles. They cannot evaluate continuous nonlinear arithmetic inequalities over coupled real-valued variables like Coulomb friction ($2 \cdot \mu \cdot F \ge m \cdot (g + a)$) or integrate differential equations over time without losing decidability. Therefore, continuous forces, accelerations, and friction thresholds must be computed outside DL by the **causal physics model / simulation engine in the Semantic Digital Twin**.

---

**E6. Strength vs. Truth in Agent Validation**

- **Logical Strength ($\text{K}_0$ vs. $\text{K}_1/\text{K}_2$):**
  - **Logical strength** measures the deductive constraining power of the agent's believed causality—how specifically its axioms can answer competency queries by ruling out logically impossible worlds.
  - **$\text{K}_0$ (No Friction Law):** Logically weak. When asked a competency query such as whether a specific grasp force $F$ will hold a container of mass $m$ under upward acceleration $a$ or allow it to slip, $\text{K}_0$ cannot entail either outcome—both slipping and holding remain logically consistent with $\text{K}_0$.
  - **$\text{K}_1$ / $\text{K}_2$ (Explicit Dynamical Laws):** Logically stronger. $\text{K}_1$ adds a static friction constraint ($2 \cdot \mu \cdot F \ge m \cdot g$) and $\text{K}_2$ adds the full dynamic acceleration law ($2 \cdot \mu \cdot F \ge m \cdot (g + a)$). Because they encode explicit causal constraints between $F, \mu, m, g,$ and $a$, they prune invalid models and produce definite entailments on competency queries (e.g., $\text{K}_2$ entails that a grasp force sufficient at rest will still slip when upward acceleration $a$ exceeds the threshold).
- **Testing for Physical Truth:**
  - Logical strength is purely an internal logical property of the axiom set: a theory can be logically strong and self-consistent yet physically false (for example, $\text{K}_1$ ignores inertial acceleration $m \cdot a$, or $\text{K}_2$ might use an inaccurate friction coefficient $\mu$).
  - To validate the actual **physical truth** of the agent's believed causality, we must run **empirical physical experiments** on the real robot and environment $[B \bowtie W]$ (varying grasp forces, payloads, and accelerations) and compare the theory's entailed predictions against observed real-world execution outcomes.

## Part 2: Robot Motion Problem and Description Logics (1.5 Pt)

**Title:** Formulating and Querying the Robot Motion Problem with Knowledge Bases & Description Logics

![Alternative Text](PR2.png)

### Scenario Background
Using the interactive Jupyter notebook and RViz simulation environment (`bmp.rviz`, `setup_demo()`), the PR2 mobile manipulator interacts with the EASE apartment model (including the fridge and various kitchen drawers). Instead of hardcoding trajectories, the robotic executive queries a knowledge base (represented via Description Logics / Prolog reasoning) to link task requests to state changes, causes (motions), and capabilities.

[Go to the programming Lab](https://binder.intel4coro.de/v2/gh/maltehue/giskard-examples/bmp_vrb?urlpath=/lab/tree/notebooks/bodyMotionProblem.ipynb)
---

### Multi-Question Student Task Sheet (0.25 Pt x 6)

1. **E7 · Knowledge Base Grounding & Task Requests**
   - Execute the initial query `taskRequest(Request, StateChange).`Examine the resulting solution bindings (e.g., `{'Request': 'open_fridge', 'StateChange': 'open(fridge)'}` and various drawer operations). Explain how mapping abstract task requests to discrete state changes forms the top-level specification layer for the manipulation executive.

2. **E8 · Extending Queries for Causal Motions & Parameters**
   - Extend the query to incorporate motion causes: `taskRequest(Request, StateChange), causes(Motion, StateChange, MotionParam).` Explain how the reasoner bridges the gap between a desired symbolic state change and continuous motion parameters (such as `envJointGoal`, target joint angles, and handle identifiers).

3. **E9 · Integrating Robot Capabilities (`canPerform`)**
   - Finalize the query by adding the robot capability constraint: `taskRequest(Request, StateChange), causes(Motion, StateChange, MotionParam), canPerform(Robot, Motion, MotionParam).`Describe what role the `Robot` variable (`'pr2'`) plays in filtering feasible trajectories through the reasoner.

4. **E10 · Specializing Request Scope (`open_fridge` vs. `open_anything`)**
   - Modify the query request parameter from general categories like `open_anything` to specifically target `open_fridge`. Explain how refining the scope in the knowledge base reduces the solution search tree and prevents unauthorized or unintended state changes in the kitchen environment.

5. **E11 · Visualization and Simulation Linkage in RViz**
   - With the background shell command executing `rviz -d .../bmp.rviz` and displaying the PR2 interacting with the fridge handle, explain how the motion parameters derived from the reasoner's output (`[0.8, b'fridge_door_joint', b'fridge_door_handle']`) are executed as physical joint trajectories in the simulation.

6. **E12 · Ontological Limits & Missing Dynamics**
   - While the knowledge base successfully infers *which* motion achieves a state change, explain why a pure Description Logic or symbolic Prolog reasoner cannot verify whether a specific grasp force will cause a wet carton to slip, and identify which subsystem must handle those physical calculations.

### Answers (Part 2)

**E7. Knowledge Base Grounding & Task Requests**

Executed query and solution bindings from the lab:
```python
# Query: taskRequest(Request, StateChange).
[{'Request': 'open_fridge',       'StateChange': 'open(fridge)'},
 {'Request': 'close_fridge',      'StateChange': 'closed(fridge)'},
 {'Request': 'open_drawer_sink',  'StateChange': 'open(drawer_sink)'},
 {'Request': 'close_drawer_sink', 'StateChange': 'closed(drawer_sink)'},
 {'Request': 'open_drawer_oven',  'StateChange': 'open(drawer_oven)'},
 {'Request': 'open_anything',     'StateChange': 'open(fridge)'},
 {'Request': 'open_anything',     'StateChange': 'open(drawer_sink)'},
 {'Request': 'open_anything',     'StateChange': 'open(drawer_oven)'}]
```

When running `taskRequest(Request, StateChange).`, the reasoner returns dictionary bindings linking high-level task names to symbolic world state changes, such as `{'Request': 'open_fridge', 'StateChange': 'open(fridge)'}`, as well as bindings for closing the fridge and opening/closing the kitchen drawers (plus general requests like `'open_anything'`).

This mapping serves as the top-level specification layer for the executive because it translates an abstract user or planner command (`Request`) into a clear, formal goal condition in the environment (`StateChange`). At this top layer, the executive only specifies *what* state change needs to happen in the world, completely decoupled from *how* the robot's joints or motors will move to achieve it.

---

**E8. Extending Queries for Causal Motions & Parameters**

Extended query and example solution binding:
```python
# Query: taskRequest(Request, StateChange), causes(Motion, StateChange, MotionParam).
{'Request': 'open_fridge',
 'StateChange': 'open(fridge)',
 'Motion': 'envJointGoal',
 'MotionParam': [0.8, b'fridge_door_joint', b'fridge_door_handle']}
```

When we extend the query to `taskRequest(Request, StateChange), causes(Motion, StateChange, MotionParam).`, the reasoner unifies the symbolic `StateChange` variable (e.g. `'open(fridge)'`) with the `causes/3` predicate.

This bridges the gap between symbolic reasoning and continuous motion control:
- `Motion` is bound to `'envJointGoal'`, which tells the executive what type of motion controller template to use (manipulating an articulated environment joint).
- `MotionParam` is bound to concrete parameters `[0.8, b'fridge_door_joint', b'fridge_door_handle']`. Here, the qualitative symbol `'open(fridge)'` is grounded into continuous and geometric values from the kitchen model: the target joint angle `0.8` rad that counts as "open", the specific URDF joint name `fridge_door_joint` to move, and the handle frame `fridge_door_handle` that the gripper needs to grasp.

---

**E9. Integrating Robot Capabilities (`canPerform`)**

Finalized query and binding with capability constraint:
```python
# Query: taskRequest(Request, StateChange), causes(Motion, StateChange, MotionParam), canPerform(Robot, Motion, MotionParam).
{'Request': 'open_fridge',
 'StateChange': 'open(fridge)',
 'Motion': 'envJointGoal',
 'MotionParam': [0.8, b'fridge_door_joint', b'fridge_door_handle'],
 'Robot': 'pr2'}
```

By adding `canPerform(Robot, Motion, MotionParam).` to the query, the reasoner checks whether a specific robot platform (`Robot = 'pr2'`) actually has the hardware capabilities needed to carry out that motion with those parameters.

While `causes(...)` only states a general physical fact (moving `fridge_door_joint` to `0.8` via `fridge_door_handle` opens the fridge), not every robot can physically do it. Binding `Robot` to `'pr2'` filters the candidate solutions by checking in the knowledge base whether the PR2 has the required kinematic chain (arm DOFs), a compatible gripper to grasp the handle, and enough workspace reachability. Any motion that would achieve the state change in theory but cannot be executed by the PR2 hardware is filtered out.

---

**E10. Specializing Request Scope (`open_fridge` vs. `open_anything`)**

Specialized query targeting `open_fridge`:
```prolog
taskRequest('open_fridge', StateChange), causes(Motion, StateChange, MotionParam), canPerform(Robot, Motion, MotionParam).
```

If we query with a broad request like `open_anything` (or leave `Request` unbound), Prolog/DL matches every openable container in the kitchen (`open(fridge)`, `open(drawer_sink)`, `open(drawer_oven)`, etc.) and explores `causes` and `canPerform` branches for all of them.

When we specialize the query to `taskRequest('open_fridge', StateChange), ...`, the variable `StateChange` is immediately unified with `'open(fridge)'` in the first step. This prunes all drawer and cabinet branches from the search tree right away, reducing the search space. Practically, it also prevents unintended state changes in the kitchen—the executive won't accidentally pick a valid solution binding that opens a sink drawer when it was only supposed to open the fridge.

---

**E11. Visualization and Simulation Linkage in RViz**

Once the reasoner outputs `MotionParam = [0.8, b'fridge_door_joint', b'fridge_door_handle']` for `Motion = 'envJointGoal'`, these values are passed to the Giskard motion controller running alongside RViz (`bmp.rviz`):
1. Using the environment URDF and TF tree, the controller looks up the 3D pose of `fridge_door_handle` and the rotation axis of `fridge_door_joint`.
2. Giskard sets up a whole-body continuous motion optimization problem: it constrains the PR2's gripper to grasp and follow `fridge_door_handle`, sets a goal constraint to drive `fridge_door_joint` to the target angle of `0.8` radians, and respects the PR2's joint limits and collision constraints.
3. The solver computes continuous joint velocity commands at each timestep and publishes the updated joint states to ROS (`/joint_states` and `/tf`), which makes the simulated PR2 in RViz reach out, grasp the fridge handle, and pull the door open to `0.8` rad.

---

**E12. Ontological Limits & Missing Dynamics**

A pure Description Logic or symbolic Prolog reasoner works with discrete symbols, static class hierarchies, and kinematic labels (like joint names, handle frames, and target angles). It cannot verify whether a grasp force will make a wet carton slip because slip depends on continuous, real-time physical dynamics: the reduced friction coefficient $\mu$ caused by water on the carton surface, the carton's mass and acceleration during motion ($2 \cdot \mu \cdot F \ge m(g + a)$), contact force distribution, and the fact that squeezing a wet cardboard carton too hard can deform or crush it.

These physical calculations must instead be handled by:
1. The **causal physics simulator / dynamics engine in the Semantic Digital Twin** (to simulate contact forces and friction limits before/during planning), and
2. The **low-level real-time force/tactile feedback controller** on the robot (which monitors slip and adjusts gripper force during actual execution).

# Submission guidelines

- Deadline: 24.11.2026, 4 PM
- Submission as Git repo commits pushed within deadlines (If not yet, create a repo with the name **iis-assignment-win26** for the whole semester and invite me as collaborator)
- Put the textual answers in notebook and all related media (images, rules, etc) should then be put into a folder called **Assignment3**. 